# Out-of-fold daily LightGBM scores on Kaggle CPU

Payee risk score for the transfer-time task (`python -m earlymule.models.oof_scores train`). For each of the 5 folds, the daily LightGBM (73 features, every positive and 1% of negatives, seed 0, early stopping on validation PR-AUC) is trained on the train rows of the other folds and scores every train row of its own fold. Folds put all members of a laundering case together, so a payee's score never comes from a model that saw its case. CPU only, so it does not use the weekly GPU quota. Each fold writes `oof_train_fold<k>.parquet` and `oof_fold<k>.json` to `/kaggle/working/` as soon as it ends.

Inputs: the export `verzol/earlymule-80k-earlymule-mule` and the fold file `verzol/earlymule-80k-oof-folds` (made locally by `python -m earlymule.models.oof_scores folds`).

In [ ]:
# Run parameters: folds to train, seed, commit to check out (None = latest main)
FOLDS = [0, 1, 2, 3, 4]
SEED = 0
COMMIT = None

In [ ]:
# Clone the repository at COMMIT and install it without dependencies (LightGBM, pandas and PyYAML are preinstalled)
!git clone -q https://github.com/Verzol/earlymule-tg.git /tmp/earlymule-tg
if COMMIT:
    !git -C /tmp/earlymule-tg checkout -q {COMMIT}
!git -C /tmp/earlymule-tg log -1 --format='%h %s'
!pip install -q -e /tmp/earlymule-tg --no-deps
!python -c "import lightgbm, pandas; print('lightgbm', lightgbm.__version__, 'pandas', pandas.__version__)"
!nproc; free -g

The export folder is found by its manifest and the fold file by name, wherever Kaggle mounts them. Each fold is its own process, so its memory is released before the next one starts.

In [ ]:
# Train every fold in order; a failed fold is logged and the next one still starts
import json, subprocess, time
from pathlib import Path
data = next(m.parent for m in Path('/kaggle/input').rglob('manifest.json')
            if json.loads(m.read_text())['dataset'] == '80K-earlymule-mule')
folds_csv = next(Path('/kaggle/input').rglob('oof_folds_80K-earlymule-mule.csv'))
out = Path('/kaggle/working')
codes = []
for k in FOLDS:
    t0 = time.time()
    cmd = ['python', '-m', 'earlymule.models.oof_scores', 'train', '--data', str(data), '--folds', str(folds_csv),
           '--fold', str(k), '--seed', str(SEED), '--out', str(out)]
    code = subprocess.call(cmd, cwd='/tmp/earlymule-tg', stdout=open(out / f'fold{k}.log', 'w'), stderr=subprocess.STDOUT)
    codes.append((k, code, round((time.time() - t0) / 60, 1)))
    print(codes[-1], flush=True)
print(codes)

In [ ]:
# Last lines of every fold log
for log in sorted(Path('/kaggle/working').glob('fold*.log')):
    print(f'--- {log.name}')
    print(log.read_text()[-1500:])